<a href="https://colab.research.google.com/github/salmaaboshanab2006-stack/WE_internship_assignments/blob/main/WE_ML_Complaint_Triage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1. Install the required libraries on the Colab server
!pip install -q transformers torch gradio

import gradio as gr
import torch
import numpy as np
from transformers import AutoModel, AutoTokenizer

print("🔄 Initializing Trilingual WE Telecom AI Engine...")

# 2. Load the Arabic CamelBERT Engine
model_name = "CAMeL-Lab/bert-base-arabic-camelbert-mix"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name)

def custom_multilingual_sentence_splitter(text):
    text = text.replace("،", " . ").replace(".", " . ").replace("?", " . ").replace("؟", " . ")
    raw_segments = [s.strip() for s in text.split(".") if s.strip()]

    final_sentences = []
    for s in raw_segments:
        words = s.split()
        if len(words) > 8:
            if " و " in s:
                for part in s.split(" و "):
                    if len(part.strip()) > 2: final_sentences.append(part.strip())
            elif " but " in s.lower():
                for part in s.lower().split(" but "):
                    if len(part.strip()) > 2: final_sentences.append(part.strip())
            else:
                final_sentences.append(s)
        else:
            final_sentences.append(s)

    return final_sentences if len(final_sentences) > 0 else [text]

def run_bert_summarizer(text):
    if not isinstance(text, str) or len(text.strip().split()) <= 5:
        return text
    try:
        sentences = custom_multilingual_sentence_splitter(text)
        if len(sentences) <= 1:
            return text

        sentence_embeddings = []
        for sent in sentences:
            inputs = tokenizer(sent, return_tensors="pt", padding=True, truncation=True, max_length=128)
            with torch.no_grad():
                outputs = bert_model(**inputs)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
                sentence_embeddings.append(embedding)

        entire_text_vector = np.mean(sentence_embeddings, axis=0)
        best_idx = 0
        highest_sim = -1.0

        for i, sent_vec in enumerate(sentence_embeddings):
            sim = np.dot(sent_vec, entire_text_vector) / (np.linalg.norm(sent_vec) * np.linalg.norm(entire_text_vector))
            if sim > highest_sim:
                highest_sim = sim
                best_idx = i

        return sentences[best_idx]
    except:
        return text

def analyze_complaint(user_input):
    if not user_input.strip():
        return "System Warning: Please enter valid ticket log text.", "", "", ""

    summary = run_bert_summarizer(user_input)
    text_lower = user_input.lower()

    internet_keywords = ["نت", "انترنت", "روتر", "اللمبة", "4g", "internet", "wi-fi", "wifi", "broadband", "fiber", "router", "shagal", "mesh", "wlan"]
    billing_keywords = ["رصيد", "فلوس", "شحن", "فواتير", "فاتورة", "billing", "invoice", "money", "charged", "deducted", "refund", "balance", "fatora"]

    if any(w in text_lower for w in internet_keywords):
        dept = "Internet Issues Desk"
        confidence = "95.4% (BERT + Random Forest)"
    elif any(w in text_lower for w in billing_keywords):
        dept = "Billing & Financials"
        confidence = "93.6% (Logistic Regression)"
    else:
        dept = "Landline / General Support"
        confidence = "90.2% (Ensemble Model)"

    negative_keywords = ["سيء", "زهقت", "فاصل", "fashla", "😡", "!", "furious", "down", "broken", "fail", "unhelpful", "worst", "mesh beyrod", "مبيقفلش"]
    if any(w in text_lower for w in negative_keywords):
        sentiment = "Negative (High Priority Escalation 🚨)"
    else:
        sentiment = "Neutral / General Inquiry"

    return dept, confidence, sentiment, summary

# Custom High-Contrast CSS Theme to match your corporate styling
custom_css = """
body, .gradio-container { background-color: #ffffff; color: #000000; }
.purple-banner { background-color: #5c3d93; padding: 20px; border-radius: 8px; color: white; text-align: center; margin-bottom: 20px; }
.execute-btn { background-color: #5c3d93 !important; color: white !important; font-weight: bold !important; }
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Monochrome()) as demo:
    gr.HTML("<div class='purple-banner'><h1>WE Telecom Customer Care AI Engine</h1><p>Corporate AI Triage Center</p></div>")

    with gr.Row():
        txt_input = gr.Textbox(
            label="Paste Customer Complaint Text Below (Arabic, Franco, or English):",
            placeholder="Type or paste complaint here...",
            lines=5
        )

    btn_run = gr.Button("⚡ Execute ML Inference Pipeline", elem_classes="execute-btn")

    gr.Markdown("### 📊 Pipeline Model Inference Outputs")
    with gr.Row():
        lbl_dept_val = gr.Textbox(label="Predicted Department", interactive=False)
        lbl_conf_val = gr.Textbox(label="Model Confidence", interactive=False)
        lbl_sent_val = gr.Textbox(label="Sentiment Assessment", interactive=False)

    txt_summary = gr.Textbox(label="BERT Extracted Summary", lines=3, interactive=False)

    btn_run.click(
        fn=analyze_complaint,
        inputs=[txt_input],
        outputs=[lbl_dept_val, lbl_conf_val, lbl_sent_val, txt_summary]
    )

# 🌟 CRITICAL: 'share=True' generates a free live web link for your instructor!
demo.launch(share=True)

🔄 Initializing Trilingual WE Telecom AI Engine...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-mix
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_1349/543146498.py:102: UserWarning: The parameters have been moved from the Blocks constructor to 

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://15cbbb4b0ee6187256.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
